# EcoPack Predictor Accuracy

This notebook evaluates the first prediction-accuracy metric for EcoPack's normalized-runtime predictor.

For each application and GPU count, it reproduces the predictor used in `ecoPack.py`:

Then it compares:
- predicted normalized runtime: `max(predictor_value) / predictor_value`
- true normalized runtime: `runtime / min(runtime)`

Main metrics in this first version:
- MAE
- RMSE


In [2]:
from pathlib import Path
import math
import re

import pandas as pd

DATA_ROOT = Path('/home/ac.zzheng/power/GPGPU/coSched/data')
SYSTEMS = ['V100', 'A100', 'H100']
SECTION_RE = re.compile(r'^===== .*?/([^/ ]+) =====$')
SM_OVERRIDE_APPS = {
    'simpleCUBLASXT',
    'simpleCUFFT_2d_MGPU',
    'simpleCUFFT_MGPU',
    'miniweather',
}


In [3]:
def parse_perf_metrics(metrics_path: Path):
    rows = []
    current_app = None

    for raw_line in metrics_path.read_text().splitlines():
        line = raw_line.strip()
        if not line:
            continue

        match = SECTION_RE.match(line)
        if match:
            current_app = match.group(1)
            continue

        if current_app is None or line.startswith('cap=') or line.startswith('gpu_count'):
            continue

        parts = line.split()
        if len(parts) < 6:
            raise ValueError(f'Expected 6 columns in {metrics_path}, got {len(parts)} for line: {line}')

        rows.append({
            'app': current_app,
            'gpu_count': int(parts[0]),
            'runtime_s': float(parts[1]),
            'avg_power_w': float(parts[2]),
            'dram_sum': float(parts[3]),
            'sm_sum': float(parts[4]),
            'fp_sum': float(parts[5]),
        })

    return pd.DataFrame(rows).sort_values(['app', 'gpu_count']).reset_index(drop=True)


def predictor_name_and_values(app_df: pd.DataFrame):
    app = app_df['app'].iloc[0]

    sm_per_gpu = {
        int(row.gpu_count): row.sm_sum / float(row.gpu_count)
        for row in app_df.itertuples(index=False)
    }
    if app in SM_OVERRIDE_APPS:
        return 'sm_sum/gpu_count (override)', sm_per_gpu

    dram_values = dict(zip(app_df['gpu_count'], app_df['dram_sum']))
    if any(value > 0.0 for value in dram_values.values()):
        return 'dram_sum', dram_values

    fp_per_gpu = {
        int(row.gpu_count): row.fp_sum / float(row.gpu_count)
        for row in app_df.itertuples(index=False)
    }
    if any(value > 0.0 for value in fp_per_gpu.values()):
        return 'fp_sum/gpu_count', fp_per_gpu

    return 'sm_sum/gpu_count', sm_per_gpu


def predicted_norm_runtime(predictor_values):
    max_value = max(predictor_values.values())
    min_value = min(predictor_values.values())

    if max_value <= 0.0:
        return {gpu_count: 1.0 for gpu_count in predictor_values}

    if abs(max_value - min_value) < 1e-12:
        return {gpu_count: 1.0 for gpu_count in predictor_values}

    result = {}
    for gpu_count, value in predictor_values.items():
        if value <= 0.0:
            raise ValueError(f'Predictor value must be positive, got {value} for {gpu_count} GPUs')
        result[gpu_count] = max_value / value
    return result


def build_prediction_frame(system: str):
    metrics_path = DATA_ROOT / system / 'perf_metrics.txt'
    df = parse_perf_metrics(metrics_path)
    rows = []

    for app, app_df in df.groupby('app', sort=True):
        predictor_name, predictor_values = predictor_name_and_values(app_df)
        pred_nrt = predicted_norm_runtime(predictor_values)
        min_runtime = app_df['runtime_s'].min()

        for row in app_df.itertuples(index=False):
            true_nrt = row.runtime_s / min_runtime
            pred = pred_nrt[int(row.gpu_count)]
            rows.append({
                'system': system,
                'app': app,
                'gpu_count': int(row.gpu_count),
                'runtime_s': row.runtime_s,
                'true_norm_runtime': true_nrt,
                'pred_norm_runtime': pred,
                'abs_error': abs(pred - true_nrt),
                'sq_error': (pred - true_nrt) ** 2,
                'predictor': predictor_name,
            })

    return pd.DataFrame(rows).sort_values(['app', 'gpu_count']).reset_index(drop=True)


In [4]:
prediction_frames = {system: build_prediction_frame(system) for system in SYSTEMS}
all_predictions = pd.concat(prediction_frames.values(), ignore_index=True)

summary_rows = []
for system, frame in prediction_frames.items():
    summary_rows.append({
        'system': system,
        'num_apps': frame['app'].nunique(),
        'num_points': len(frame),
        'mae': frame['abs_error'].mean(),
        'rmse': math.sqrt(frame['sq_error'].mean()),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('system').reset_index(drop=True)
summary_df


,system,num_apps,num_points,mae,rmse
0,A100,19,69,1.229893,5.334963
1,H100,19,69,2.639181,17.100235
2,V100,17,61,0.237126,0.396639


In [6]:
app_summary_rows = []
for system, frame in prediction_frames.items():
    grouped = frame.groupby('app', sort=True)
    for app, app_df in grouped:
        app_summary_rows.append({
            'system': system,
            'app': app,
            'num_points': len(app_df),
            'predictor': app_df['predictor'].iloc[0],
            'mae': app_df['abs_error'].mean(),
            'rmse': math.sqrt(app_df['sq_error'].mean()),
        })

app_summary_df = pd.DataFrame(app_summary_rows).sort_values(['system', 'rmse', 'app']).reset_index(drop=True)
# app_summary_df


In [7]:
for system in SYSTEMS:
    print(f'===== {system} =====')
    display_cols = ['app', 'gpu_count', 'predictor', 'true_norm_runtime', 'pred_norm_runtime', 'abs_error']
    display(prediction_frames[system][display_cols].round(4))


===== V100 =====


,app,gpu_count,predictor,true_norm_runtime,pred_norm_runtime,abs_error
0,MonteCarloMultiGPU,1,sm_sum/gpu_count,1.0000,1.0445,0.0445
1,MonteCarloMultiGPU,2,sm_sum/gpu_count,1.0023,1.0384,0.0362
2,MonteCarloMultiGPU,3,sm_sum/gpu_count,1.0013,1.0316,0.0304
3,MonteCarloMultiGPU,4,sm_sum/gpu_count,1.0296,1.0000,0.0296
4,bert,1,dram_sum,2.6806,3.9496,1.2690
...,...,...,...,...,...,...
56,vgg16,4,dram_sum,1.0764,1.8281,0.7517
57,vgg19,1,dram_sum,3.0288,2.4025,0.6262
58,vgg19,2,dram_sum,1.3236,1.8202,0.4966
59,vgg19,3,dram_sum,1.0961,1.5641,0.4681


===== A100 =====


,app,gpu_count,predictor,true_norm_runtime,pred_norm_runtime,abs_error
0,MonteCarloMultiGPU,1,sm_sum/gpu_count,1.0000,1.0090,0.0090
1,MonteCarloMultiGPU,2,sm_sum/gpu_count,1.0621,1.0037,0.0584
2,MonteCarloMultiGPU,3,sm_sum/gpu_count,1.0744,1.0128,0.0616
3,MonteCarloMultiGPU,4,sm_sum/gpu_count,1.0747,1.0000,0.0747
4,bert,1,dram_sum,2.0582,3.7500,1.6918
...,...,...,...,...,...,...
64,vgg16,4,dram_sum,1.1083,1.9000,0.7917
65,vgg19,1,dram_sum,1.0000,1.1364,0.1364
66,vgg19,2,dram_sum,1.1049,1.0000,0.1049
67,vgg19,3,dram_sum,1.1136,1.1905,0.0768


===== H100 =====


,app,gpu_count,predictor,true_norm_runtime,pred_norm_runtime,abs_error
0,MonteCarloMultiGPU,1,fp_sum/gpu_count,1.0000,1.0000,0.0000
1,MonteCarloMultiGPU,2,fp_sum/gpu_count,1.0020,1.0000,0.0020
2,MonteCarloMultiGPU,3,fp_sum/gpu_count,1.0718,1.0449,0.0269
3,MonteCarloMultiGPU,4,fp_sum/gpu_count,1.1636,1.0783,0.0853
4,bert,1,dram_sum,1.8041,3.4980,1.6939
...,...,...,...,...,...,...
64,vgg16,4,dram_sum,1.1866,1.0756,0.1110
65,vgg19,1,dram_sum,1.0000,1.0077,0.0077
66,vgg19,2,dram_sum,1.0719,1.0785,0.0066
67,vgg19,3,dram_sum,1.1299,1.0000,0.1299


In [ ]:
import matplotlib.pyplot as plt

mae_violin_df = app_summary_df[['system', 'app', 'mae']].copy()
system_order = ['V100', 'A100', 'H100']
data = [mae_violin_df.loc[mae_violin_df['system'] == system, 'mae'].values for system in system_order]

fig, ax = plt.subplots(figsize=(8, 5))
parts = ax.violinplot(data, showmeans=True, showmedians=True, showextrema=True)

colors = ['#4C78A8', '#F58518', '#54A24B']
for body, color in zip(parts['bodies'], colors):
    body.set_facecolor(color)
    body.set_edgecolor('black')
    body.set_alpha(0.65)

for key in ['cmeans', 'cmedians', 'cbars', 'cmins', 'cmaxes']:
    if key in parts:
        parts[key].set_color('black')
        parts[key].set_linewidth(1.0)

for idx, system in enumerate(system_order, start=1):
    subset = mae_violin_df[mae_violin_df['system'] == system].sort_values('mae', ascending=False)
    top = subset.head(3)
    ax.scatter([idx] * len(subset), subset['mae'], color='black', s=12, alpha=0.35, zorder=3)
    for _, row in top.iterrows():
        ax.annotate(
            row['app'],
            (idx, row['mae']),
            xytext=(5, 0),
            textcoords='offset points',
            fontsize=8,
            va='center'
        )

ax.set_xticks(range(1, len(system_order) + 1))
ax.set_xticklabels(system_order)
ax.set_ylabel('Per-app MAE of normalized runtime prediction')
ax.set_title('EcoPack Predictor Accuracy Distribution by System')
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

mae_violin_df.sort_values(['system', 'mae'], ascending=[True, False]).reset_index(drop=True)
